In [3]:
import ast
import numpy as np
import matplotlib.pyplot as plt

from collections import Counter
from itertools import combinations
from igraph import Graph

# ============================================================
# CONFIGURATION
# ============================================================

RULE_FILE = "../17_NuevoAnalisisP1/reglas_abac_karimi.txt"
THRESHOLD = 0.25
OUTPUT_GRAPH = "../23_PolicyNetworkAnalysis/rule_network_amz_k.graphml"

# ============================================================
# LOAD RULES
# ============================================================

def rule_to_txt(rule):
    str_rule = ""
    for i in rule:
        str_rule += str(i) + ","
    return str_rule

# ============================================================
# CENTRALITY METRICS AS NODE ATTRIBUTES
# ============================================================

def add_centrality_attributes(g):

    print("\nComputing centrality metrics...")

    # --------------------------------------------------------
    # Degree
    # --------------------------------------------------------

    degree_values = g.degree()

    g.vs["degree"] = degree_values

    # --------------------------------------------------------
    # Betweenness
    # --------------------------------------------------------

    betweenness_values = g.betweenness()

    g.vs["betweenness"] = betweenness_values

    # --------------------------------------------------------
    # Closeness
    # --------------------------------------------------------

    closeness_values = g.closeness()

    g.vs["closeness"] = closeness_values

    # --------------------------------------------------------
    # Eigenvector Centrality
    # --------------------------------------------------------

    eigenvector_values = g.eigenvector_centrality(
        weights=g.es["weight"]
    )

    g.vs["eigenvector"] = eigenvector_values

    # --------------------------------------------------------
    # Pagerank (recommended)
    # --------------------------------------------------------

    pagerank_values = g.pagerank(
        weights=g.es["weight"]
    )

    g.vs["pagerank"] = pagerank_values

    # --------------------------------------------------------
    # Local Clustering Coefficient
    # --------------------------------------------------------

    clustering_values = g.transitivity_local_undirected()

    g.vs["local_clustering"] = clustering_values

    print("Centrality attributes added!")

    return g

def load_rules(file_path):

    rules = []

    with open(file_path, "r", encoding="utf-8") as f:

        for idx, line in enumerate(f):

            line = line.strip()

            if not line:
                continue

            try:

                parsed = ast.literal_eval(line)

                id_com = parsed[0][1]

                attributes = parsed[1]

                feature_set = set()

                for attr_name, attr_value in attributes:
                    feature_set.add((attr_name, str(attr_value)))

                rules.append({
                    "idx": idx,
                    "id_com": id_com,
                    "features": feature_set,
                    "raw": parsed
                })

            except Exception as e:
                print(f"Error parsing line {idx}: {e}")

    return rules


# ============================================================
# JACCARD SIMILARITY
# ============================================================

def jaccard_similarity(set_a, set_b):

    intersection = len(set_a.intersection(set_b))
    union = len(set_a.union(set_b))

    if union == 0:
        return 0.0

    return intersection / union


# ============================================================
# BUILD NETWORK
# ============================================================

def build_rule_network(rules, threshold):

    g = Graph()
    g.add_vertices(len(rules))

    # --------------------------------------------------------

    for rule in rules:

        v = rule["idx"]

        g.vs[v]["rule_id"] = v
        g.vs[v]["id_com"] = rule["id_com"]
        g.vs[v]["num_features"] = len(rule["features"])
        g.vs[v]["rule"] = rule_to_txt(rule["raw"])

    # --------------------------------------------------------

    edges = []
    weights = []

    for r1, r2 in combinations(rules, 2):

        sim = jaccard_similarity(
            r1["features"],
            r2["features"]
        )

        if sim > threshold:

            edges.append((r1["idx"], r2["idx"]))
            weights.append(sim)

    g.add_edges(edges)
    g.es["weight"] = weights

    return g


# ============================================================
# BASIC ANALYSIS
# ============================================================

def analyze_network(g):

    print("\n================================")
    print("RULE NETWORK ANALYSIS")
    print("================================")

    print(f"Nodes: {g.vcount()}")
    print(f"Edges: {g.ecount()}")

    density = g.density()
    print(f"Density: {density:.6f}")

    degrees = g.degree()

    print(f"Average degree: {np.mean(degrees):.4f}")
    print(f"Max degree: {np.max(degrees)}")

    clustering = g.transitivity_avglocal_undirected()
    print(f"Average clustering coefficient: {clustering:.4f}")

    comps = g.connected_components()

    print(f"Connected components: {len(comps)}")
    print(f"Giant component size: {max(comps.sizes())}")

    # --------------------------------------------------------
    # Average shortest path length
    # --------------------------------------------------------

    giant = comps.giant()

    avg_shortest = giant.average_path_length()

    print(f"Average shortest path length: {avg_shortest:.4f}")

    # --------------------------------------------------------
    # Community detection
    # --------------------------------------------------------

    communities = g.community_multilevel(weights=g.es["weight"])

    modularity = communities.modularity

    print(f"Communities detected: {len(communities)}")
    print(f"Modularity: {modularity:.4f}")

    print("================================\n")


# ============================================================
# PLOT: EDGE WEIGHT DISTRIBUTION
# ============================================================

def plot_edge_weight_distribution(g):

    if g.ecount() == 0:
        print("No edges to plot.")
        return

    weights = g.es["weight"]

    plt.figure(figsize=(8, 5))

    plt.hist(weights, bins=30)

    plt.xlabel("Edge Weight (Jaccard Similarity)")
    plt.ylabel("Frequency")
    plt.title("Edge Weight Distribution")

    plt.grid(True)

    plt.tight_layout()
    plt.show()


# ============================================================
# PLOT: DEGREE DISTRIBUTION
# ============================================================

def plot_degree_distribution(g):

    degrees = g.degree()

    degree_count = Counter(degrees)

    x = list(degree_count.keys())
    y = list(degree_count.values())

    plt.figure(figsize=(8, 5))

    plt.scatter(x, y)

    plt.xlabel("Degree")
    plt.ylabel("Frequency")
    plt.title("Degree Distribution")

    plt.grid(True)

    plt.tight_layout()
    plt.show()


# ============================================================
# CENTRALITY ANALYSIS
# ============================================================

def compute_centralities(g):

    centralities = {}

    # --------------------------------------------------------
    # Degree Centrality
    # --------------------------------------------------------

    centralities["degree"] = g.degree()

    # --------------------------------------------------------
    # Betweenness Centrality
    # --------------------------------------------------------

    centralities["betweenness"] = g.betweenness()

    # --------------------------------------------------------
    # Closeness Centrality
    # --------------------------------------------------------

    centralities["closeness"] = g.closeness()

    # --------------------------------------------------------
    # Eigenvector Centrality
    # --------------------------------------------------------

    centralities["eigenvector"] = g.eigenvector_centrality(
        weights=g.es["weight"]
    )

    return centralities


# ============================================================
# PLOT CENTRALITY DISTRIBUTIONS
# ============================================================

def plot_centrality_distribution(values, title):

    plt.figure(figsize=(8, 5))

    plt.hist(values, bins=30)

    plt.xlabel("Centrality Value")
    plt.ylabel("Frequency")
    plt.title(title)

    plt.grid(True)

    plt.tight_layout()
    plt.show()


# ============================================================
# VISUALIZE NETWORK USING CENTRALITY
# ============================================================

def visualize_network_by_centrality(g, values, title):

    layout = g.layout("fr")

    # --------------------------------------------------------
    # Normalize node size
    # --------------------------------------------------------

    values = np.array(values)

    if np.max(values) == np.min(values):
        node_sizes = [20] * len(values)
    else:
        node_sizes = 10 + (
            (values - np.min(values))
            / (np.max(values) - np.min(values))
        ) * 40

    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(10, 10))

    igraph_plot = g.copy()

    visual_style = {
        "layout": layout,
        "vertex_size": node_sizes,
        "vertex_label": None,
        "edge_width": 0.3,
        "bbox": (800, 800),
        "margin": 40,
        "target": ax
    }

    from igraph import plot

    plot(igraph_plot, **visual_style)

    plt.title(title)

    plt.show()


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    print("Loading rules...")
    rules = load_rules(RULE_FILE)

    print(f"Rules loaded: {len(rules)}")
    for r in rules:
        print(r)

    print("Building rule network...")
    G = build_rule_network(rules, THRESHOLD)

    # print(G)

    # --------------------------------------------------------

    # analyze_network(G)

    # --------------------------------------------------------
    # PLOTS
    # --------------------------------------------------------

    # plot_edge_weight_distribution(G)

    # plot_degree_distribution(G)

    # --------------------------------------------------------
    # CENTRALITIES
    # --------------------------------------------------------

    centralities = compute_centralities(G)

    # --------------------------------------------------------
    # DISTRIBUTIONS
    # --------------------------------------------------------

    # plot_centrality_distribution(
    #     centralities["degree"],
    #     "Degree Centrality Distribution"
    # )

    # plot_centrality_distribution(
    #     centralities["betweenness"],
    #     "Betweenness Centrality Distribution"
    # )

    # plot_centrality_distribution(
    #     centralities["closeness"],
    #     "Closeness Centrality Distribution"
    # )

    # plot_centrality_distribution(
    #     centralities["eigenvector"],
    #     "Eigenvector Centrality Distribution"
    # )

    # --------------------------------------------------------

    # --------------------------------------------------------
    # ADD CENTRALITY ATTRIBUTES
    # --------------------------------------------------------

    G = add_centrality_attributes(G)
    

    # --------------------------------------------------------
    # SAVE GRAPH
    # --------------------------------------------------------

    G.write_graphml(OUTPUT_GRAPH)

    print(f"\nGraph saved to: {OUTPUT_GRAPH}")

Loading rules...
Rules loaded: 18
{'idx': 0, 'id_com': '0', 'features': {('ROLE_FAMILY_DESC', '-117906'), ('ROLE_FAMILY', '-290919'), ('ROLE_ROLLUP_1', '-117961')}, 'raw': [['id_com', '0'], [['ROLE_ROLLUP_1', -117961], ['ROLE_FAMILY_DESC', -117906], ['ROLE_FAMILY', -290919]]]}
{'idx': 1, 'id_com': '0', 'features': {('ROLE_ROLLUP_2', '118327'), ('ROLE_CODE', '118322'), ('ROLE_TITLE', '118321'), ('ROLE_FAMILY', '290919')}, 'raw': [['id_com', '0'], [['ROLE_ROLLUP_2', 118327], ['ROLE_TITLE', 118321], ['ROLE_FAMILY', 290919], ['ROLE_CODE', 118322]]]}
{'idx': 2, 'id_com': '0', 'features': {('ROLE_FAMILY', '118638'), ('ROLE_FAMILY', '-290919'), ('ROLE_ROLLUP_2', '118343')}, 'raw': [['id_com', '0'], [['ROLE_ROLLUP_2', 118343], ['ROLE_FAMILY', 118638], ['ROLE_FAMILY', -290919]]]}
{'idx': 3, 'id_com': '0', 'features': {('ROLE_FAMILY', '118424'), ('ROLE_ROLLUP_2', '118343')}, 'raw': [['id_com', '0'], [['ROLE_ROLLUP_2', 118343], ['ROLE_FAMILY', 118424]]]}
{'idx': 4, 'id_com': '0', 'features': {('R

In [4]:
for i, g in enumerate(G.connected_components().subgraphs()):
    print(" ======== ", i , " ========")
    for v in g.vs():
        print(v["rule_id"], v["rule"])

 ========  0  ========
0 ['id_com', '0'],[['ROLE_ROLLUP_1', -117961], ['ROLE_FAMILY_DESC', -117906], ['ROLE_FAMILY', -290919]],
14 ['id_com', '0'],[['ROLE_ROLLUP_2', 118327], ['ROLE_FAMILY_DESC', -117906], ['ROLE_FAMILY', -290919]],
 ========  1  ========
1 ['id_com', '0'],[['ROLE_ROLLUP_2', 118327], ['ROLE_TITLE', 118321], ['ROLE_FAMILY', 290919], ['ROLE_CODE', 118322]],
11 ['id_com', '0'],[['ROLE_ROLLUP_2', 118225], ['ROLE_DEPTNAME', 120551], ['ROLE_TITLE', 118321], ['ROLE_FAMILY', 290919], ['ROLE_CODE', 118322]],
15 ['id_com', '0'],[['ROLE_TITLE', 118321], ['ROLE_FAMILY_DESC', 117906], ['ROLE_CODE', 118322]],
 ========  2  ========
2 ['id_com', '0'],[['ROLE_ROLLUP_2', 118343], ['ROLE_FAMILY', 118638], ['ROLE_FAMILY', -290919]],
 ========  3  ========
3 ['id_com', '0'],[['ROLE_ROLLUP_2', 118343], ['ROLE_FAMILY', 118424]],
 ========  4  ========
4 ['id_com', '0'],[['ROLE_ROLLUP_1', 117961], ['ROLE_ROLLUP_2', 118300]],
6 ['id_com', '0'],[['ROLE_ROLLUP_1', 117961], ['ROLE_ROLLUP_2', 118